# Chapter 9: Multi-Agent Systems


In [ ]:
from dotenv import load_dotenv
load_dotenv()

%load_ext autoreload
%autoreload 2


## 9.1 Sequential Workflow


In [ ]:
from scratch_agent.workflows.sequential import SequentialWorkflow
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient

researcher = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    instructions="You are a research assistant. Summarize findings concisely in 2-3 sentences.",
    name="researcher",
)
writer = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    instructions="You are a technical writer. Polish and format the content you receive. Keep it brief.",
    name="writer",
)

workflow = SequentialWorkflow(agents=[researcher, writer])
result = await workflow.run("Explain how transformers work in deep learning.")
print(f"Output: {result.output}")
print(f"Steps: {result.context.current_step}")

## 9.2 Parallel Workflow


In [ ]:
from scratch_agent.workflows.parallel import ParallelWorkflow
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient

analyst_a = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    instructions="Analyze from a technical perspective. Be concise (2-3 sentences).",
    name="technical_analyst",
)
analyst_b = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    instructions="Analyze from a business perspective. Be concise (2-3 sentences).",
    name="business_analyst",
)

workflow = ParallelWorkflow(agents=[analyst_a, analyst_b])
result = await workflow.run("Evaluate the pros and cons of microservices architecture.")
print(result.output)

## 9.3 Loop Workflow


In [ ]:
from scratch_agent.workflows.loop import LoopWorkflow
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.context import AgentResult

refiner = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    instructions="You iteratively improve text. When you're satisfied with the quality, include 'FINAL:' before your final version.",
    name="refiner",
)

def stop_condition(result: AgentResult, iteration: int) -> bool:
    """Stop when the agent signals completion or max iterations."""
    return result.output and "FINAL:" in str(result.output)

workflow = LoopWorkflow(agents=[refiner], stop_condition=stop_condition, max_iterations=3)
result = await workflow.run("Write a haiku about programming.")
print(f"Output: {result.output}")

## 9.4 Agent as Tool


In [ ]:
from scratch_agent.tools.agent_tool import AgentTool
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.tools.base import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

math_agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[calculator],
    instructions="You solve math problems step by step using the calculator.",
    name="math_solver",
    description="Solves math problems step by step",
)

# Wrap the math agent as a tool
math_tool = AgentTool(agent=math_agent)
print(f"AgentTool name: {math_tool.name}")
print(f"AgentTool description: {math_tool.description}")

# Create a main agent that can delegate to the math agent
main_agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[math_tool],
    instructions="You are a tutor. Use the math_solver tool for math questions.",
    max_steps=5,
)

result = await main_agent.run("What is 17 * 23 + 45?")
print(f"\nResult: {result.output}")

## 9.5 Agent Transfer


In [ ]:
from scratch_agent.agent import Agent
from scratch_agent.llm import LlmClient
from scratch_agent.tools.base import tool

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

# Create sub-agents
math_agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    tools=[calculator],
    instructions="You handle math questions using the calculator tool.",
    name="math_agent",
    description="Handles mathematical calculations",
)

general_agent = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    instructions="You handle general knowledge questions.",
    name="general_agent",
    description="Handles general knowledge and trivia",
)

# Create triage agent with sub_agents (enables transfer_to_agent tool)
triage = Agent(
    model=LlmClient(model="gpt-4o-mini"),
    instructions="You are a triage agent. Route math questions to math_agent and other questions to general_agent.",
    name="triage",
    sub_agents=[math_agent, general_agent],
    max_steps=5,
)

print(f"Triage tools: {[t.name for t in triage.tools]}")

result = await triage.run("What is 42 * 17?")
print(f"Result: {result.output}")

## 9.6 Remote Agent (A2A)


In [ ]:
from scratch_agent.remote import RemoteAgent

# Remote Agent requires a running A2A server
# Example: Start the math agent server first:
#   python -m scratch_agent.a2a_server
#
# Then connect:
# remote = RemoteAgent(base_url="http://localhost:8000")
# result = await remote.run("What is 15 + 27?")
# print(result.output)

print("RemoteAgent connects to agents via A2A (Agent-to-Agent) protocol.")
print("It discovers the agent's capabilities via .well-known/agent.json")
print("and communicates using JSON-RPC over HTTP.")